In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [0]:
# DBTITLE 1,Read Raw Data
flights_df = spark.table('workspace.default.on_time_performance_report_url')
carriers_df = spark.table('workspace.default.carriers_maps')
airports_df = spark.table('workspace.default.airports_maps')

In [0]:
# Select necessary columns for ALL analyses
columns_to_keep = [
    "FlightDate", "Reporting_Airline", "Tail_Number", "Flight_Number_Reporting_Airline",
    "Origin", "OriginCityName", "OriginState",
    "Dest", "DestCityName", "DestState",
    "CRSDepTime", "DepTime", "DepDelay", "DepDelayMinutes", "DepDel15", "DepTimeBlk",
    "TaxiOut", "WheelsOff", "WheelsOn", "TaxiIn", # For operational efficiency
    "CRSArrTime", "ArrTime", "ArrDelay", "ArrDelayMinutes", "ArrDel15", "ArrTimeBlk",
    "Cancelled", "CancellationCode", "Diverted",
    "CRSElapsedTime", "ActualElapsedTime", "AirTime", # For operational efficiency
    "CarrierDelay", "WeatherDelay", "NASDelay", "SecurityDelay", "LateAircraftDelay",
    "Year", "Month", "DayofMonth", "DayOfWeek"
]

flights_selected_df = flights_df.select(*columns_to_keep)


In [0]:
# Rename columns
flights_renamed_df = flights_selected_df \
    .withColumnRenamed("FlightDate", "flight_date") \
    .withColumnRenamed("Reporting_Airline", "carrier_code") \
    .withColumnRenamed("Origin", "origin_code") \
    .withColumnRenamed("Dest", "dest_code") \
    .withColumnRenamed("ArrDelayMinutes", "arr_delay_mins") \
    .withColumnRenamed("DepDelayMinutes", "dep_delay_mins") \
    .withColumnRenamed("Cancelled", "cancelled") \
    .withColumnRenamed("Diverted", "diverted") \
    .withColumnRenamed("CRSElapsedTime", "crs_elapsed_time") \
    .withColumnRenamed("ActualElapsedTime", "actual_elapsed_time") \
    .withColumnRenamed("AirTime", "air_time") \
    .withColumnRenamed("TaxiOut", "taxi_out") \
    .withColumnRenamed("TaxiIn", "taxi_in") \
    .withColumnRenamed("CarrierDelay", "carrier_delay") \
    .withColumnRenamed("WeatherDelay", "weather_delay") \
    .withColumnRenamed("NASDelay", "nas_delay") \
    .withColumnRenamed("SecurityDelay", "security_delay") \
    .withColumnRenamed("LateAircraftDelay", "late_aircraft_delay")

In [0]:
flights_casted_df = flights_renamed_df \
            .withColumn("flight_date", F.to_date(F.col("flight_date"))) \
            .withColumn("arr_delay_mins", F.col("arr_delay_mins").cast(DoubleType())) \
            .withColumn("dep_delay_mins", F.col("dep_delay_mins").cast(DoubleType())) \
            .withColumn("cancelled", F.col("cancelled").cast(IntegerType())) \
            .withColumn("diverted", F.col("diverted").cast(IntegerType())) \
            .withColumn("DepDel15", F.col("DepDel15").cast(IntegerType())) \
            .withColumn("ArrDel15", F.col("ArrDel15").cast(IntegerType())) \
            .withColumn("crs_elapsed_time", F.col("crs_elapsed_time").cast(DoubleType())) \
            .withColumn("actual_elapsed_time", F.col("actual_elapsed_time").cast(DoubleType())) \
            .withColumn("air_time", F.col("air_time").cast(DoubleType())) \
            .withColumn("taxi_out", F.col("taxi_out").cast(DoubleType())) \
            .withColumn("taxi_in", F.col("taxi_in").cast(DoubleType())) \
            .withColumn("carrier_delay", F.col("carrier_delay").cast(DoubleType())) \
            .withColumn("weather_delay", F.col("weather_delay").cast(DoubleType())) \
            .withColumn("nas_delay", F.col("nas_delay").cast(DoubleType())) \
            .withColumn("security_delay", F.col("security_delay").cast(DoubleType())) \
            .withColumn("late_aircraft_delay", F.col("late_aircraft_delay").cast(DoubleType()))

In [0]:
numeric_cols_to_fill = [
    "carrier_delay", "weather_delay", "nas_delay", "security_delay", "late_aircraft_delay",
    "taxi_out", "taxi_in", "air_time", "actual_elapsed_time", "crs_elapsed_time",
    "arr_delay_mins", "dep_delay_mins" 
]
# Fill delays only if 0 makes sense contextually
flights_final_df = flights_casted_df.na.fill(0, subset=numeric_cols_to_fill)

In [0]:
# Final DataFrame is ready for analysis
# Note: .cache() is not needed on serverless compute

# Preparing the final so spark can run efficiently
flights_final_df.cache()

In [0]:
# DBTITLE 1,Joining datasets...
# Prepare lookup tables
# Use get() to safely access split results, returning null for out-of-bounds indices
split_desc = F.split(F.col('Description'),":")
airports_df = airports_df.withColumn('city', F.get(split_desc, 0)).withColumn('airport_full_name', F.get(split_desc, 1))

airports_lookup = airports_df.select(
    F.col("Code").alias("airport_code"),
    F.col("city").alias("airport_city"),
    F.col("airport_full_name")
).distinct()

carriers_lookup = carriers_df.select(
    F.col("Code").alias("carrier_code_lookup"),
    F.col("Description").alias("carrier_name")
).distinct()


In [0]:
base_join = flights_final_df \
    .join(carriers_lookup, flights_final_df.carrier_code == carriers_lookup.carrier_code_lookup, "left")

origin_join = base_join \
    .join(airports_lookup.alias("origin_apt"), base_join.origin_code == F.col("origin_apt.airport_code"), "left") \
    .select(base_join["*"], carriers_lookup["carrier_name"],
            F.col("origin_apt.airport_full_name").alias("origin_airport_name"),
            F.col("origin_apt.airport_city").alias("origin_city_code"))

full_flight_data = origin_join \
    .join(airports_lookup.alias("dest_apt"), origin_join.dest_code == F.col("dest_apt.airport_code"), "left") \
    .select(origin_join["*"], # Select all from previous join
            F.col("dest_apt.airport_full_name").alias("dest_airport_name"),
            F.col("dest_apt.airport_city").alias("dest_state_code"))

# Select and potentially rename final columns for clarity
# (Example selecting a subset, adjust as needed)
full_flight_data = full_flight_data.select(
        "flight_date", "Year", "Month", "DayofMonth", "DayOfWeek",
        "carrier_code", "carrier_name",
        "Tail_Number", "Flight_Number_Reporting_Airline",
        "origin_code", "origin_airport_name", "origin_city_code",
        "dest_code", "dest_airport_name", "dest_state_code",
        "DepTimeBlk", "ArrTimeBlk",
        "dep_delay_mins", "DepDel15",
        "arr_delay_mins", "ArrDel15",
        "cancelled", "diverted", "CancellationCode",
        "carrier_delay", "weather_delay", "nas_delay", "security_delay", "late_aircraft_delay",
        "taxi_out", "taxi_in", "air_time", "actual_elapsed_time", "crs_elapsed_time"
    )

In [0]:
full_flight_data.display()

In [0]:
%sql
Create schema if not exists workspace.silver

In [0]:
full_flight_data.write.mode("overwrite").saveAsTable("workspace.silver.full_flight_data")
